# Basic FastHTML HTMX SSE Integration Example

> Demonstrates real-time tqdm progress tracking in FastHTML using HTMX with the SSE extension. Streams HTML fragments (not JSON) directly over Server-Sent Events, enabling automatic DOM updates without custom JavaScript. Features out-of-band swaps for button state management, proper error handling with timeouts, and clean job completion states. Uses DaisyUI and Tailwind for styling with helper functions for consistent UI components.

In [1]:
from fasthtml.common import *
from fasthtml.common import sse_message
import uuid, time
import asyncio
import random

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
    app.hdrs.insert(htmx_idx+2, Script(code="""window.addEventListener('beforeunload',()=>{document.querySelectorAll('[sse-connect]').forEach(e=>{e['htmx-internal-data']?.sseEventSource?.close()});htmx?.findAll('[sse-connect]').forEach(e=>htmx.trigger(e,'htmx:sseClose'))});"""))
else:
    print("HTMX not found")

In [5]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(1000), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [6]:
# Centralized helper functions to reduce duplication

# Element IDs
ELEMENT_IDS = {
    'progress_label': 'progress-label',
    'progress_bar': 'progress-bar', 
    'status_text': 'status-text',
    'progress_display': 'progress-display',
    'progress_container': 'progress-container',
    'start_btn': 'start-btn'
}

def create_progress_bar(value="0", max="100", element_id=None, **attrs):
    """Create a styled progress bar with consistent styling"""
    if element_id:
        attrs['id'] = element_id
    return Progress(
        value=str(value), 
        max=str(max), 
        cls=combine_classes(progress, progress_colors.primary, w.full),
        **attrs
    )

In [7]:
def create_progress_label(text="Progress:", element_id=None, **attrs):
    """Create a styled progress label with consistent styling"""
    if element_id:
        attrs['id'] = element_id
    return P(text, cls=combine_classes(font_weight.bold), **attrs)

In [8]:
def create_status_message(text, element_id=None, **attrs):
    """Create a styled status message with consistent styling"""
    if element_id:
        attrs['id'] = element_id
    return P(text, cls=combine_classes(m.t(2), font_size.sm), **attrs)

In [9]:
def create_progress_display(label_text="Progress:", progress_value="0", status_text="Starting...", 
                           show_progress_bar=True, use_element_ids=False, oob_swap=False, **attrs):
    """
    Create a unified progress display that can be used for both static and streaming updates.
    
    Args:
        label_text: Text for the progress label
        progress_value: Current progress value (0-100)
        status_text: Status message to display
        show_progress_bar: Whether to show the progress bar
        use_element_ids: Whether to add element IDs for SSE updates
        oob_swap: Whether to enable out-of-band swap
        **attrs: Additional attributes for the container div
    """
    children = []
    
    # Add label
    children.append(create_progress_label(
        label_text, 
        element_id=ELEMENT_IDS['progress_label'] if use_element_ids else None
    ))
    
    # Add progress bar if requested
    if show_progress_bar:
        children.append(create_progress_bar(
            value=str(progress_value),
            element_id=ELEMENT_IDS['progress_bar'] if use_element_ids else None
        ))
    
    # Add status message
    children.append(create_status_message(
        status_text,
        element_id=ELEMENT_IDS['status_text'] if use_element_ids else None
    ))
    
    # Set container attributes
    if 'id' not in attrs:
        attrs['id'] = ELEMENT_IDS['progress_display']
    
    if oob_swap:
        attrs['hx_swap_oob'] = 'true'
    
    return Div(*children, **attrs)

In [10]:
def create_sse_progress_display(job_id, initial_value="0", initial_status="Starting..."):
    """Create a progress display connected to SSE stream"""
    
    # Use the unified function with SSE attributes
    return create_progress_display(
        progress_value=initial_value,
        status_text=initial_status,
        use_element_ids=True,
        # HTMX SSE attributes
        hx_ext="sse",
        sse_connect=f"/stream?job_id={job_id}",
        sse_swap="message",
        hx_swap="innerHTML"
    )

In [11]:
def render_start_button(disabled=False, oob_swap=False):
    """Render start button with appropriate state"""
    btn_classes = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    attrs = {
        'id': ELEMENT_IDS['start_btn'],
        'cls': btn_classes,
        'disabled': disabled
    }
    
    if not disabled:
        attrs['hx_post'] = '/start_task'
        attrs['hx_target'] = f"#{ELEMENT_IDS['progress_container']}"
        attrs['hx_swap'] = 'innerHTML'
    
    if oob_swap:
        attrs['hx_swap_oob'] = 'true'
    
    return Button("Start Task", **attrs)

In [12]:
def create_final_response(label_text="Progress:", progress_value=100, status_text="Completed!", 
                         enable_button=True, is_error=False):
    """
    Create a final response with static display and OOB button update.
    
    Args:
        label_text: Text for the progress label
        progress_value: Final progress value (None for errors)
        status_text: Final status message
        enable_button: Whether to enable the start button
        is_error: Whether this is an error response
    """
    # Re-enable button via out-of-band swap
    button = render_start_button(disabled=not enable_button, oob_swap=True)
    
    # Create static display with OOB swap to replace SSE-connected div
    display = create_progress_display(
        label_text=label_text,
        progress_value=progress_value if progress_value is not None else 0,
        status_text=status_text,
        show_progress_bar=(progress_value is not None),
        use_element_ids=False,
        oob_swap=True
    )
    
    return Div(button, display)

In [13]:
@rt("/")
def index():
    return create_test_page(
        "Basic HTMX SSE Progress Demo",
        Div(
            H2("Simple Progress Tracking with HTMX SSE Extension"),
            render_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id=ELEMENT_IDS['progress_container'],
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        )
    )

In [14]:
# API endpoints using HTMX SSE
@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
        
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return SSE-connected progress display with disabled button via out-of-band swap
    return Div(
        # Disable button via out-of-band swap
        render_start_button(disabled=True, oob_swap=True),
        # Progress display with SSE connection
        create_sse_progress_display(job_id)
    )

In [15]:
def parse_progress_data(progress_data):
    """Extract progress information from SSE data"""
    progress_value = progress_data.get('progress', 0)
    
    # Build status text from bar info if available
    status_text = f"Processing... {progress_value:.1f}%"
    if progress_data.get('bars'):
        first_bar = next(iter(progress_data['bars'].values()))
        status_text = f"{first_bar['desc']}: {first_bar['pct']:.1f}% ({first_bar['cur']}/{first_bar['tot']})"
    
    return progress_value, status_text

@rt("/stream")
def stream(job_id: str):
    """SSE endpoint that streams progress updates as HTML"""
    
    async def html_stream():
        """Generate HTML updates for HTMX SSE"""
        import json
        
        try:
            # Use the async SSE stream generator
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                
                # Parse the SSE data format
                if data.startswith('data: '):
                    try:
                        # Extract JSON from SSE data line
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            
                            # Check if completed
                            if progress_data.get('completed'):
                                # Final state - use helper
                                yield sse_message(create_final_response(
                                    progress_value=100,
                                    status_text="Completed!"
                                ))
                                break
                            else:
                                # Progress update - extract data
                                progress_value, status_text = parse_progress_data(progress_data)
                                
                                # Create update using unified function
                                html_content = create_progress_display(
                                    progress_value=int(progress_value),
                                    status_text=status_text,
                                    use_element_ids=True
                                )
                                
                                yield sse_message(html_content)
                    except json.JSONDecodeError as e:
                        print(f"[STREAM] JSON decode error: {e}")
                        pass  # Skip invalid data
                        
                elif data.startswith('event: end'):
                    # Handle end event - job not found or timed out
                    print(f"[STREAM] Received 'event: end' - job not found or timed out")
                    
                    # Use helper for error response
                    yield sse_message(create_final_response(
                        label_text="Error:",
                        progress_value=None,
                        status_text="Job not found or timed out",
                        is_error=True
                    ))
                    break
                    
                elif data.startswith(': '):
                    # Heartbeat/keep-alive - pass through as-is
                    print(f"[STREAM] Sending heartbeat")
                    yield data
                    
        except Exception as e:
            print(f"[STREAM] ERROR in html_stream: {e}")
            import traceback
            traceback.print_exc()
            
            # Use helper for error response
            yield sse_message(create_final_response(
                label_text="Error:",
                progress_value=None,
                status_text=f"Stream error: {str(e)}",
                is_error=True
            ))
        
    try:
        return EventStream(html_stream())
    except Exception as e:
        print(f"[STREAM] ERROR creating EventStream: {e}")
        import traceback
        traceback.print_exc()
        raise

In [18]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX(port=server.port))

Processing: 100%|███████████████████████████| 1000/1000 [00:10<00:00, 99.05it/s]


In [19]:
# Stop server when done
server.stop()